# Phase 1 · Teacher embeddings
Build the MS MARCO data, then embed the corpus and the query sets once with the frozen teacher (Qwen3-Embedding-4B, MRL d=256, fp16) on 2 GPUs. No fine-tuning here.

Run with `DEBUG = True` first, check the last cell, then set `DEBUG = False`.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch

DEBUG = True    # first pass: small data (~10k passages), same teacher model. Set False for the full run.
NPROCS = 2      # one process per T4

# --- paths: debug and full runs get separate caches, so a debug pass never overwrites a real run
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
CACHE_DIR = f"{ROOT}/cache/{'debug' if DEBUG else 'full'}"
DATA_PATH = f"{CACHE_DIR}/data.json"
EMB_DIR = f"{CACHE_DIR}/emb/teacher"

# --- data (MS MARCO, Tevatron/msmarco-passage train split)
DATA = dict(n_rows=210_000,     # first rows of the split (streamed, cached as parquet)
            core_n=100_000,     # seeded sample of queries -> train / val / test (split by query)
            n_val=5_000, n_test=5_000,
            n_align=200_000,    # alignment queries = train + other queries, never val/test
            split_seed=1234,
            use_title=False)    # passage text = "title. text" when True
if DEBUG:
    DATA.update(n_rows=12_000, core_n=5_000, n_val=500, n_test=500, n_align=10_000)

# --- teacher embedding (data-parallel, one shard per GPU, resumable)
TEACHER = dict(model="Qwen/Qwen3-Embedding-4B", dim=256, dtype="fp16",
               doc_max_len=128, query_max_len=64, batch_size=64,
               data_path=DATA_PATH, emb_dir=EMB_DIR)
CHECKS = dict(sanity_n=1000,    # passages for the fp16-vs-fp32 check
              timing_n=2000,    # passages timed for the projection
              budget_h=5.0)     # only reported: shrink DATA yourself if over

## 1 · Build data
Rows → by-query splits → corpus (positives + one stored negative per core query, deduplicated) → alignment queries. Cached to `data.json`.

In [ ]:
from data import load_rows, build_data, save_json, load_json

if os.path.exists(DATA_PATH):
    data = load_json(DATA_PATH)
    print("[data] cached")
else:
    rows = load_rows(DATA["n_rows"], CACHE_DIR)
    data = build_data(rows, **{k: v for k, v in DATA.items() if k != "n_rows"})
    save_json(data, DATA_PATH)
    print("one row:", {k: (v if k == "query" else str(v)[:120]) for k, v in rows[0].items()})
print({k: len(v["ids"]) for k, v in data.items()})

## 2 · Pre-flight
fp16 vs fp32 on a passage sample (must have no NaNs), then a timing projection for the full run.

In [ ]:
from model import get_device
from teacher import fp16_sanity, time_embedding, free_gpu_memory

device = get_device()
if os.path.exists(f"{EMB_DIR}/corpus.npy"):
    print("corpus already embedded, pre-flight skipped")
else:
    sanity = fp16_sanity(TEACHER["model"], data["corpus"]["texts"][:CHECKS["sanity_n"]], TEACHER["dim"],
                         TEACHER["doc_max_len"], device, device_map_fp32="auto" if torch.cuda.device_count() > 1 else None)
    assert sanity["nan_rows"] == 0, "fp16 teacher produced NaNs"
    time_embedding(TEACHER, data, device, world_size=NPROCS, n_sample=CHECKS["timing_n"], budget_h=CHECKS["budget_h"])
    free_gpu_memory()

## 3 · Embed on both GPUs
Rank r embeds shard r of every set. The corpus gets no prefix; queries get the instruction prompt. Finished shards are skipped, so re-running after a restart resumes.

In [ ]:
from ddp import launch
from teacher import embed_main

launch(embed_main, TEACHER, nprocs=NPROCS)

## 4 · Inspect
Every row should have norm ≈ 1.0 and the ids should match `data.json`.

In [ ]:
import numpy as np
from data import load_emb

for name in ("corpus", "align", "val", "test"):
    emb, ids = load_emb(EMB_DIR, name)
    norms = np.linalg.norm(emb.astype(np.float32), axis=1)
    size_mb = os.path.getsize(f"{EMB_DIR}/{name}.npy") / 1e6
    print(f"{name:<7} {str(emb.shape):<15} {emb.dtype}  norm min/max {norms.min():.4f}/{norms.max():.4f}  "
          f"{size_mb:8.1f} MB  ids match: {ids == data[name]['ids']}")